 # 03 Final Load Prep
Use this notebook to compute final KPIs, prepare the Tableau-ready dataset, and export the exact file used in the dashboard.



In [13]:
import pandas as pd

master = pd.read_csv("../data/processed/cleaned_master.csv")
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'shipping_limit_date'
]
for col in date_cols:
    master[col]=pd.to_datetime(master[col])

print(f"Loaded—{master.shape[0]:,}rows,{master.shape[1]} columns")

Loaded—114,092rows,43 columns


In [14]:
total_revenue= master['total_item_value'].sum()
total_orders= master['order_id'].nunique()
total_items_sold= len(master)
aov= total_revenue / total_orders  # Average Order Value
print("=" * 45)
print("          REVENUE KPIs")
print("=" * 45)
print(f"  Total Revenue        : {total_revenue:>15,.2f}")
print(f"  Total Orders         : {total_orders:>15,}")
print(f"  Total Items Sold     : {total_items_sold:>15,}")
print(f"  Avg Order Value      : {aov:>15,.2f}")
print("=" * 45)

          REVENUE KPIs
  Total Revenue        :   15,915,872.32
  Total Orders         :          99,441
  Total Items Sold     :         114,092
  Avg Order Value      :          160.05


In [15]:
delivered = master[master['order_status'] == 'delivered'].copy()

avg_delivery_days  = delivered['delivery_time_days'].mean()
median_delivery    = delivered['delivery_time_days'].median()
late_orders        = delivered['is_late'].sum()
total_delivered    = delivered['order_id'].nunique()
late_rate          = (late_orders / total_delivered) * 100
on_time_rate       = 100 - late_rate

print("=" * 45)
print("          DELIVERY KPIs")
print("=" * 45)
print(f"  Total Delivered Orders : {total_delivered:>12,}")
print(f"  Avg Delivery Time      : {avg_delivery_days:>12.1f} days")
print(f"  Median Delivery Time   : {median_delivery:>12.1f} days")
print(f"  Late Orders            : {int(late_orders):>12,}")
print(f"  Late Delivery Rate     : {late_rate:>12.1f}%")
print(f"  On-Time Delivery Rate  : {on_time_rate:>12.1f}%")
print("=" * 45)

          DELIVERY KPIs
  Total Delivered Orders :       96,478
  Avg Delivery Time      :         12.0 days
  Median Delivery Time   :         10.0 days
  Late Orders            :        8,755
  Late Delivery Rate     :          9.1%
  On-Time Delivery Rate  :         90.9%


In [16]:
total_customers= master['customer_unique_id'].nunique()
repeat_customers= master.groupby('customer_unique_id')['order_id'].nunique()
repeat_count= (repeat_customers > 1).sum()
repeat_rate= (repeat_count / total_customers) * 100
top_state= master.groupby('customer_state')['order_id'].nunique().idxmax()
top_city= master.groupby('customer_city')['order_id'].nunique().idxmax()

print("=" * 45)
print("          CUSTOMER KPIs")
print("=" * 45)
print(f"  Total Unique Customers : {total_customers:>12,}")
print(f"  Repeat Customers       : {repeat_count:>12,}")
print(f"  Repeat Purchase Rate   : {repeat_rate:>12.1f}%")
print(f"  Top Customer State     : {top_state:>12}")
print(f"  Top Customer City      : {top_city:>12}")
print("=" * 45)

          CUSTOMER KPIs
  Total Unique Customers :       96,096
  Repeat Customers       :        2,997
  Repeat Purchase Rate   :          3.1%
  Top Customer State     :           SP
  Top Customer City      :    sao paulo


In [17]:
total_products= master['product_id'].nunique()
total_sellers= master['seller_id'].nunique()
total_categories= master['product_category_name_english'].nunique()
top_category= (master.groupby('product_category_name_english')['total_item_value'].sum().idxmax())
top_seller= (master.groupby('seller_id')['total_item_value'].sum().idxmax())
avg_price= master['price'].mean()
avg_freight= master['freight_value'].mean()

print("=" * 45)
print("       PRODUCT & SELLER KPIs")
print("=" * 45)
print(f"  Total Products         : {total_products:>12,}")
print(f"  Total Sellers          : {total_sellers:>12,}")
print(f"  Total Categories       : {total_categories:>12,}")
print(f"  Top Category (Revenue) : {top_category:>12}")
print(f"  Avg Item Price         : BRL {avg_price:>10,.2f}")
print(f"  Avg Freight Value      : BRL {avg_freight:>10,.2f}")
print("=" * 45)

       PRODUCT & SELLER KPIs
  Total Products         :       32,951
  Total Sellers          :        3,095
  Total Categories       :           71
  Top Category (Revenue) : health_beauty
  Avg Item Price         : BRL     120.48
  Avg Freight Value      : BRL      19.98


In [19]:
avg_review = master['review_score'].mean()
pct_5_star = (master['review_score'] == 5).sum() / master['review_score'].count() * 100
pct_1_star = (master['review_score'] == 1).sum() / master['review_score'].count() * 100
score_dist = master['review_score'].value_counts().sort_index()

print("=" * 45)
print("       SATISFACTION KPIs")
print("=" * 45)
print(f"  Avg Review Score  : {avg_review:>15.2f} / 5")
print(f"  5-Star Rate       : {pct_5_star:>15.1f}%")
print(f"  1-Star Rate       : {pct_1_star:>15.1f}%")
print("=" * 45)
print("\n  Score Distribution:")
for score, count in score_dist.items():
    bar = "█" * int(count / 2000)
    print(f"  {score}★  {count:>6,}  {bar}")

       SATISFACTION KPIs
  Avg Review Score  :            4.02 / 5
  5-Star Rate       :            56.2%
  1-Star Rate       :            13.1%

  Score Distribution:
  1.0★  14,775  ███████
  2.0★   3,936  █
  3.0★   9,476  ████
  4.0★  21,348  ██████████
  5.0★  63,596  ███████████████████████████████


In [20]:
payment_split = (master.groupby('payment_type')['total_item_value']
                 .sum()
                 .sort_values(ascending=False)
                 .reset_index())
payment_split['revenue_pct'] = (payment_split['total_item_value'] /
                                 payment_split['total_item_value'].sum() * 100).round(1)

avg_installments = master['payment_installments'].mean()
top_payment      = payment_split.iloc[0]['payment_type']

print("=" * 45)
print("       PAYMENT KPIs")
print("=" * 45)
print(f"  Top Payment Method    : {top_payment:>12}")
print(f"  Avg Installments      : {avg_installments:>12.1f}")
print("\n  Revenue by Payment Type:")
print(payment_split.to_string(index=False))
print("=" * 45)

       PAYMENT KPIs
  Top Payment Method    :  credit_card
  Avg Installments      :          3.1

  Revenue by Payment Type:
payment_type  total_item_value  revenue_pct
 credit_card       12504817.04         78.6
      boleto        2859446.84         18.0
     voucher         335625.92          2.1
  debit_card         215839.06          1.4
 not_defined              0.00          0.0


In [22]:
kpi_summary = pd.DataFrame([
    # Revenue
    {"Category": "Revenue",      "KPI": "Total Revenue (BRL)",        "Value": round(total_revenue, 2)},
    {"Category": "Revenue",      "KPI": "Total Orders",                "Value": total_orders},
    {"Category": "Revenue",      "KPI": "Total Items Sold",            "Value": total_items_sold},
    {"Category": "Revenue",      "KPI": "Avg Order Value (BRL)",       "Value": round(aov, 2)},
    # Delivery
    {"Category": "Delivery",     "KPI": "Total Delivered Orders",      "Value": total_delivered},
    {"Category": "Delivery",     "KPI": "Avg Delivery Time (days)",    "Value": round(avg_delivery_days, 1)},
    {"Category": "Delivery",     "KPI": "On-Time Delivery Rate (%)",   "Value": round(on_time_rate, 1)},
    {"Category": "Delivery",     "KPI": "Late Delivery Rate (%)",      "Value": round(late_rate, 1)},
    # Customers
    {"Category": "Customers",    "KPI": "Total Unique Customers",      "Value": total_customers},
    {"Category": "Customers",    "KPI": "Repeat Customers",            "Value": repeat_count},
    {"Category": "Customers",    "KPI": "Repeat Purchase Rate (%)",    "Value": round(repeat_rate, 1)},
    {"Category": "Customers",    "KPI": "Top Customer State",          "Value": top_state},
    # Products
    {"Category": "Products",     "KPI": "Total Products",              "Value": total_products},
    {"Category": "Products",     "KPI": "Total Sellers",               "Value": total_sellers},
    {"Category": "Products",     "KPI": "Total Categories",            "Value": total_categories},
    {"Category": "Products",     "KPI": "Top Revenue Category",        "Value": top_category},
    {"Category": "Products",     "KPI": "Avg Item Price (BRL)",        "Value": round(avg_price, 2)},
    # Satisfaction
    {"Category": "Satisfaction", "KPI": "Avg Review Score",            "Value": round(avg_review, 2)},
    {"Category": "Satisfaction", "KPI": "5-Star Rate (%)",             "Value": round(pct_5_star, 1)},
    {"Category": "Satisfaction", "KPI": "1-Star Rate (%)",             "Value": round(pct_1_star, 1)},
    # Payments
    {"Category": "Payments",     "KPI": "Top Payment Method",          "Value": top_payment},
    {"Category": "Payments",     "KPI": "Avg Installments",            "Value": round(avg_installments, 1)},
])

kpi_summary.to_csv("../data/processed/kpi_summary.csv", index=False)

print("KPI Summary saved to data/processed/kpi_summary.csv\n")
print(kpi_summary.to_string(index=False))

KPI Summary saved to data/processed/kpi_summary.csv

    Category                       KPI         Value
     Revenue       Total Revenue (BRL)   15915872.32
     Revenue              Total Orders         99441
     Revenue          Total Items Sold        114092
     Revenue     Avg Order Value (BRL)        160.05
    Delivery    Total Delivered Orders         96478
    Delivery  Avg Delivery Time (days)          12.0
    Delivery On-Time Delivery Rate (%)          90.9
    Delivery    Late Delivery Rate (%)           9.1
   Customers    Total Unique Customers         96096
   Customers          Repeat Customers          2997
   Customers  Repeat Purchase Rate (%)           3.1
   Customers        Top Customer State            SP
    Products            Total Products         32951
    Products             Total Sellers          3095
    Products          Total Categories            71
    Products      Top Revenue Category health_beauty
    Products      Avg Item Price (BRL)        